# PDF Loader

In [ ]:
from pathlib import Path
from typing import List

import fitz

from src.models.document import Document, DocumentMetadata

def load_pdf(file_path: Path) -> List[Document]:
    """ 
    Load a PDF file and return a list of Document objects, 
    where each Document represents a single page. 
    """
    pdf_path = Path(file_path)

    # validate file existence
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")
    
    # validate file extension
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Expected a PDF file, got: {pdf_path.suffix}")
    
    documents: List[Document] = []

    # Open pdf
    pdf_document = fitz.open(pdf_path)

    try:
        # iterate through each page
        for page_index in range(len(pdf_document)):
            page = pdf_document[page_index]

            # extract raw text
            text = page.get_text()

            #text_blocks = page.get_text("blocks")
            #text_blocks.sort(key=lambda b: (b[0], b[1]))

            # concatenate text from all blocks
            #text = "\n\n ".join([block[4].strip() for block in text_blocks if block[4].strip()])

            # skip empty pages
            if not text.strip(): 
                continue

            # create metadata object
            metadata = DocumentMetadata(
                source=pdf_path.name,
                file_type="pdf",
                page=page_index + 1,
            )

            # create document object
            document = Document(
                page_content=text,
                metadata=metadata
            )

            documents.append(document)
    
    finally: 
        pdf_document.close()

    return documents 

# PDF Cleaner

In [ ]:
import re
from typing import List
from src.models.document import Document

class TextCleaner:
    def __init__(self):
        # 1. Regex to strip out the web-print date and header boilerplate
        self.header_boilerplate = re.compile(
            r"^\s*\d{1,2}/\d{1,2}/\d{2,4},\s*\d{1,2}:\d{2}\s*(?:AM|PM)\s*|MacBook Air \(13-inch, M5\) - Tech Specs - Apple Support \(IN\)"
        )
        
        # 2. Regex to strip out footer links, URLs, and page counters (e.g., 1/7)
        self.footer_url_pattern = re.compile(r"https://support\.apple\.com/[^\s]+")
        self.page_counter_pattern = re.compile(r"\b\d\s*/\s*\d\b")
        
        # 3. Strip out the parser omission comments
        self.omitted_pictures = re.compile(r"\*\*==>\s*picture\s*\[.*?\]\s*intentionally\s*omitted\s*<==\*\*")
        
        # 4. Dictionary of known corrupted inline broken words found in Apple's specs layout
        self.word_fixes = {
            r"\bAccessi\s+bility\b": "Accessibility",
            r"\bCon\s+fig\s+ure\b": "Configure",
            r"\bcon\s+fig\s+ure\b": "configure",
            r"\bEn\s+viron\s+men\s+tal\b": "Environmental",
            r"\ben\s+viron\s+men\s+tal\b": "environmental",
        }

    def clean_text(self, text: str) -> str:
        """Applies normalization steps to a single string block."""
        if not text:
            return ""

        # Remove image omission tags
        text = self.omitted_pictures.sub("", text)

        # Split lines to clean up per-page headers/footers cleanly
        lines = text.splitlines()
        cleaned_lines = []

        for line in lines:
            # Clean header metadata noise out of the line text
            line_cleaned = self.header_boilerplate.sub("", line)
            line_cleaned = self.footer_url_pattern.sub("", line_cleaned)
            line_cleaned = self.page_counter_pattern.sub("", line_cleaned)
            
            cleaned_lines.append(line_cleaned)

        # Recombine lines
        text = "\n".join(cleaned_lines)

        # Fix specific broken words using regex keys
        for broken_pattern, fixed_word in self.word_fixes.items():
            text = re.sub(broken_pattern, fixed_word, text)

        # Remove web interactive feedback artifact left at the bottom of the scrape
        text = re.sub(r"\*\*Helpful\?\*\*\s*Yes\s*No.*", "", text, flags=re.DOTALL)
        text = re.sub(r"Copyright\s*©\s*2026\s*Apple\s*Inc\..*", "", text, flags=re.DOTALL)

        # Collapse multiple empty newlines down to a max of two to keep markdown paragraphs distinct
        text = re.sub(r"\n{3,}", "\n\n", text)
        
        return text.strip()


def preprocess_documents(documents: List[Document]) -> List[Document]:
    """
    Main entry point for the preprocessing module pipeline.
    Iterates over parsed documents, cleans their content layer, 
    and returns sanitized Document objects ready for chunking.
    """
    cleaner = TextCleaner()
    preprocessed_docs: List[Document] = []

    for doc in documents:
        cleaned_content = cleaner.clean_text(doc.page_content)
        
        # Keep the page object if it still holds functional text data after cleaning
        if cleaned_content:
            # Create a clean instance inheriting metadata parameters
            cleaned_doc = Document(
                page_content=cleaned_content,
                metadata=doc.metadata
            )
            preprocessed_docs.append(cleaned_doc)

    return preprocessed_docs

In [1]:
from langchain_docling.loader import DoclingLoader

loader = DoclingLoader(file_path="../documents/pdfs/MacBook Air (13-inch, M5) - Tech Specs.pdf")

w:\Data-Science\Projects\8.FullStack-RAG-Application-Project-Atman\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
w:\Data-Science\Projects\8.FullStack-RAG-Application-Project-Atman\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dhanush\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either ne

In [4]:
docs = loader.load()

In [16]:
docs[1].page_content

'Configurable to:\nSky\xa0Blue\nSilver\nStarlight\nMidnight\n10-core CPU with 4\xa0super cores and 6\xa0efficiency cores\n8-core GPU, 10-core GPU\nNeural Accelerators\nHardware-accelerated ray tracing\n16-core Neural\xa0Engine\n153GB/s memory bandwidth\nHardware-accelerated H.264, HEVC, ProRes and ProRes\xa0RAW\nVideo decode engine\nVideo encode engine\nProRes encode and decode engine\nAV1 decode\nM5 with 10-core\xa0CPU and 10-core\xa0GPU\nMacBook Air (13-inch, M5) - Tech Specs - Apple Support (IN)'

# Chunker

In [ ]:
from pathlib import Path
from typing import List

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from src.models.document import Document, DocumentMetadata

class MarkdownLayoutSplitter:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        """
        Initializes a structure-aware Markdown splitter tailored for RAG architectures.
        """
        # Define the structural Markdown headers we want to split by
        self.headers_to_split_on = [
            ("#", "title"),
            ("##", "section"),
            ("###", "sub_section")
        ]
        
        self.markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=self.headers_to_split_on,
            strip_headers=False  # Keep headers within text so LLMs maintain semantic context
        )
        
        # Fallback recursive splitter if a single markdown section is larger than our target chunk window
        self.recursive_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def split_documents(self, cleaned_documents: List[Document]) -> List[Document]:
        """
        Processes cleaned source documents, dynamically tracks hierarchical section headings,
        and splits them into text windows ready for vectorization.
        """
        final_chunks: List[Document] = []

        for doc in cleaned_documents:
            # 1. Structural splits based on markdown heading layers
            markdown_sections = self.markdown_splitter.split_text(doc.page_content)

            for section in markdown_sections:
                # Extract headers mapped by MarkdownHeaderTextSplitter
                section_metadata = section.metadata
                current_section_name = section_metadata.get("section", section_metadata.get("title", None))

                # 2. Sub-chunking if a single spec block exceeds the chunk size limit
                sub_text_chunks = self.recursive_splitter.split_text(section.page_content)

                for sub_text in sub_text_chunks:
                    if not sub_text.strip():
                        continue

                    # Create a deep copy of your original document metadata tracking
                    chunk_metadata = DocumentMetadata(
                        source=doc.metadata.source,
                        file_type=doc.metadata.file_type,
                        page=doc.metadata.page,
                        section=current_section_name  # Injects the active header name directly into metadata
                    )

                    final_chunks.append(
                        Document(
                            page_content=sub_text.strip(),
                            metadata=chunk_metadata
                        )
                    )

        return final_chunks

# New codes

In [18]:
from pathlib import Path
from typing import List

import pymupdf4llm

from src.models.document import Document, DocumentMetadata


def load_pdfs(pdf_directory: Path) -> List[Document]:
    """
    Load all PDFs from a directory.

    Returns:
        List[Document]
    """

    pdf_directory = Path(pdf_directory)

    if not pdf_directory.exists():
        raise FileNotFoundError(
            f"Directory not found: {pdf_directory}"
        )

    pdf_files = sorted(pdf_directory.glob("*.pdf"))

    if not pdf_files:
        raise ValueError(
            f"No PDF files found in {pdf_directory}"
        )

    documents: List[Document] = []

    for pdf_path in pdf_files:

        pages_data = pymupdf4llm.to_markdown(
            str(pdf_path),
            page_chunks=True
        )

        for page in pages_data:

            text = page["text"]

            if not text.strip():
                continue

            metadata = DocumentMetadata(
                source=pdf_path.name,
                file_type="pdf",
                page=page["metadata"]["page_number"]
            )

            documents.append(
                Document(
                    page_content=text,
                    metadata=metadata
                )
            )

    return documents

In [19]:
pdf_docs = load_pdfs("../documents/pdfs")

In [26]:
len(pdf_docs)

53

In [27]:
pdf_docs[35:53]

[Document(page_content='5/28/26, 11:50 AM \n\niPhone 17 Pro - Tech Specs - Apple Support (IN) \n\nEnglish (Australia, UK, US), Chinese (Simplified, Traditional, Traditional – Hong Kong), French (Canada, France), German, Italian, Japanese, Korean, Spanish (Latin America, Spain), Arabic, Bulgarian, Catalan, Croatian, Czech, Danish, Dutch, Finnish, Greek, Hebrew, Hindi, Hungarian, Indonesian, Kazakh, Malay, Norwegian, Polish, Portuguese (Brazil, Portugal), Romanian, Russian, Slovak, Swedish, Thai, Turkish, Ukrainian, Vietnamese \n\n## Learn more about feature availability \n\n## **In the Box** \n\n## **iPhone 17 Pro** \n\n## **USB-C Charge Cable (1m)** \n\n## **Documentation** \n\nAs part of our efforts to reach carbon neutrality by 2030, iPhone 17 Pro and iPhone 17 Pro Max do not include a power adapter or EarPods. Included in the box is a USB-C Charge Cable that supports fast charging and is compatible with USB-C power adapters and computer ports. \n\nWe encourage you to use any compati

In [1]:
import pandas as pd
df = pd.read_csv("document_chunks.csv")

In [ ]:
df['chunk_index'].

0      0
1      1
2     10
3     11
4     12
5     13
6     14
7     15
8     16
9     17
10    18
11    19
12     2
13    20
14     3
15     4
16     5
17     6
18     7
19     8
Name: chunk_index, dtype: int64

In [4]:
df.head(20)

,id,text_content,embedding,source,file_type,page,section,chunk_index,parent_document_id
0,apollo_11_chunk_0,"Apollo 11 (July 16–24, 1969) was the American ...","[-0.03329522,0.014396063,-0.0054187193,-0.0631...",Apollo 11.txt,txt,NaN,NaN,0,apollo_11
1,apollo_11_chunk_1,Launched atop a Saturn V rocket from Kennedy S...,"[-0.02012112,0.009340887,0.0077717747,-0.07057...",Apollo 11.txt,txt,NaN,NaN,1,apollo_11
2,apollo_11_chunk_10,"Background In the late 1950s and early 1960s, ...","[-0.0049308906,0.020120759,0.014781034,-0.0785...",Apollo 11.txt,txt,NaN,NaN,10,apollo_11
3,apollo_11_chunk_11,. Its success demonstrated that the USSR could...,"[0.013191731,0.01955225,-0.0019515236,-0.06242...",Apollo 11.txt,txt,NaN,NaN,11,apollo_11
4,apollo_11_chunk_12,This incident sparked the Sputnik crisis and i...,"[-0.010395891,0.031098839,0.011987174,-0.07405...",Apollo 11.txt,txt,NaN,NaN,12,apollo_11
5,apollo_11_chunk_13,". Nearly a month later, on May 5, 1961, Alan S...","[-0.02890695,0.018986722,0.0035135776,-0.08811...",Apollo 11.txt,txt,NaN,NaN,13,apollo_11
6,apollo_11_chunk_14,Because the Soviet Union had launch vehicles w...,"[-0.010694185,0.02296151,0.00642217,-0.0611303...",Apollo 11.txt,txt,NaN,NaN,14,apollo_11
7,apollo_11_chunk_15,I believe that this nation should commit itsel...,"[-0.026857147,0.013182955,0.015062608,-0.07027...",Apollo 11.txt,txt,NaN,NaN,15,apollo_11
8,apollo_11_chunk_16,. We propose to accelerate the development of ...,"[-0.009013045,0.0040808367,0.014352995,-0.0601...",Apollo 11.txt,txt,NaN,NaN,16,apollo_11
9,apollo_11_chunk_17,". But in a very real sense, it will not be one...","[-0.023581427,-0.0014182315,0.030396488,-0.068...",Apollo 11.txt,txt,NaN,NaN,17,apollo_11


# Testing with gemini-2.5-flash

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

 

from src.core.embedding import GeminiEmbeddingGenerator
from src.core.generator import Generator
from src.core.reranker import CohereReranker
from src.core.retriever import Retriever
from src.core.vector_store import VectorStore
from src.utils.config import EMBEDDING_MODEL, LLM_MODEL, RERANKER_MODEL, TOP_K, TOP_N


embedding_generator = GeminiEmbeddingGenerator(model_name=EMBEDDING_MODEL)
vector_store = VectorStore()
reranker = CohereReranker(model_name=RERANKER_MODEL, top_n=TOP_N)
retriever = Retriever(
    embedder=embedding_generator,
    vector_store=vector_store,
    top_k=TOP_K,
    reranker=reranker,
)
generator = Generator(model_name="gemini-2.5-flash")

In [29]:
query = "What three structural components does NemoClaw combine to transition OpenClaw into a controlled sandbox"

In [30]:
retrieved_docs = retriever.retrieve(query=query)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query, retrieved_chunks=retrieved_docs
)

answer

'NemoClaw combines a host CLI, sandbox plugin, and versioned blueprint to move OpenClaw into a controlled sandbox.\nSource: NemoClaw Architecture Overview.md, Section: NemoClaw Architecture Overview'

In [45]:
query1 = "Where does the NemoClaw gateway structurally sit, and what boundary does OpenShell enforce"

In [46]:
retrieved_docs = retriever.retrieve(query=query1)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query1, retrieved_chunks=retrieved_docs
)

answer

'The NemoClaw gateway sits between NemoClaw control, the sandbox, inference providers, and external integrations. OpenShell enforces the sandbox boundary.\nSource: NemoClaw Architecture Overview.md, Section: High-Level Flow'

In [33]:
query2 = "When a user wants an always-on assistant with standardized NVIDIA defaults and minimal assembly, should they prefer NemoClaw or standalone OpenShell"

In [34]:
retrieved_docs = retriever.retrieve(query=query2)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query2, retrieved_chunks=retrieved_docs
)

answer

'When a user wants an always-on assistant with standardized NVIDIA defaults and minimal assembly, they should prefer NemoClaw.\n\nSource: NemoClaw Ecosystem.md, When to Use Which'

In [35]:
query3 = "In NemoClaw release v0.0.53, what environment variable can be configured if a user intentionally wants a fresh workspace recreated without initiating an automated preflight backup"

In [36]:
retrieved_docs = retriever.retrieve(query=query3)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query3, retrieved_chunks=retrieved_docs
)

answer

'To recreate a fresh workspace without initiating an automated preflight backup in NemoClaw v0.0.53, you can set the environment variable `NEMOCLAW_RECREATE_WITHOUT_BACKUP=1`. (Source: NemoClaw Release Notes.md, Section: v0.0.53)'

In [37]:
query4 = "What were the individual launch components of the Apollo 11 spacecraft, and which specific component was designed to return to Earth"

In [38]:
retrieved_docs = retriever.retrieve(query=query4)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query4, retrieved_chunks=retrieved_docs
)

answer

'The Apollo spacecraft consisted of three parts: the command module (CM), the service module (SM), and the Lunar Module (LM) (Source: Apollo 11.txt, Chunk 1).\n\nThe command module (CM) was the only part designed to return to Earth (Source: Apollo 11.txt, Chunk 1).'

In [39]:
query5 = "How much time passed between Neil Armstrong becoming the first human to walk on the Moon and Buzz Aldrin following him onto the surface"

In [40]:
retrieved_docs = retriever.retrieve(query=query5)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query5, retrieved_chunks=retrieved_docs
)

answer

'Buzz Aldrin followed Neil Armstrong onto the lunar surface nineteen minutes after Armstrong became the first human to walk on the Moon. (Source: Apollo 11.txt, Chunk 1)'

In [41]:
query6 = "What directive formally established the Artemis program, what year was it signed, and by which United States president?"

In [42]:
retrieved_docs = retriever.retrieve(query=query6)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query6, retrieved_chunks=retrieved_docs
)

answer

'The Artemis program was formally established via Space Policy Directive-1 in 2017 by President Donald Trump.\nSource: Artemis Program.txt, Page: None, Section: None'

In [43]:
query7 = "What are the three pieces put together in a NemoClaw deployment, and what is the distinct functional scope or responsibility assigned to each"

In [44]:
retrieved_docs = retriever.retrieve(query=query7)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query7, retrieved_chunks=retrieved_docs
)

answer

"The three pieces put together in a NemoClaw deployment are OpenClaw, OpenShell, and NemoClaw, each with a distinct scope (Source: NemoClaw Ecosystem.md, Section: How the Stack Fits Together).\n\nNemoClaw sits above OpenShell in the operator workflow, driving OpenShell APIs and CLI to create and configure the sandbox that runs OpenClaw. Models and endpoints sit behind OpenShell's inference routing (Source: NemoClaw Ecosystem.md, Section: How the Stack Fits Together).\n\nThe responsibilities:\n*   **OpenClaw**: Runs inside the sandbox with the NemoClaw plugin (Source: NemoClaw Architecture Overview.md, Section: High-Level Flow).\n*   **OpenShell**: Owns sandbox lifecycle, networking, policy enforcement, inference routing, and integration egress (Source: NemoClaw Architecture Overview.md, Section: High-Level Flow).\n*   **NemoClaw**: Collects configuration, runs onboarding, prepares the blueprint, and asks OpenShell to create or update resources (Source: NemoClaw Architecture Overview.md

In [47]:
query8 = "To what mock address does an agent inside a NemoClaw sandbox route its inference traffic, and does the sandbox receive the host's actual API keys?"

In [48]:
retrieved_docs = retriever.retrieve(query=query8)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query8, retrieved_chunks=retrieved_docs
)

answer

"An agent inside a NemoClaw sandbox routes its inference traffic to `inference.local`. The sandbox does not receive the host's actual API keys; provider credentials stay on the host.\n\nSource: NemoClaw Architecture Overview.md, Section: Inference Routing; NemoClaw Inference Options.md, Section: How Inference Routing Works"

In [49]:
query9 = "What environment variable and specific hardware are required to activate the experimental Local NVIDIA NIM provider option in NemoClaw?"

In [50]:
retrieved_docs = retriever.retrieve(query=query9)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query9, retrieved_chunks=retrieved_docs
)

answer

'To activate the experimental Local NVIDIA NIM provider option in NemoClaw, the environment variable `NEMOCLAW_EXPERIMENTAL=1` is required, along with a NIM-capable GPU.\n\nSource: NemoClaw Inference Options.md, Section: Provider Status'

In [51]:
query10 = "Under what scenario does the Local vLLM option automatically appear in the onboarding menu without requiring any experimental configuration flags"

In [52]:
retrieved_docs = retriever.retrieve(query=query10)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query10, retrieved_chunks=retrieved_docs
)

answer

'The Local vLLM option automatically appears in the onboarding menu without requiring any experimental configuration flags when NemoClaw detects a vLLM server already running on `localhost:8000`.\n\nSource: NemoClaw Inference Options.md, Section: Provider Status, Chunk 1; Section: Provider Options, Chunk 2; Section: Caveated Local Options, Chunk 3 and 5.'

In [53]:
query11 = "On what specific platform infrastructures can self-evolving claws run safely using NemoClaw's security guardrails"

In [54]:
retrieved_docs = retriever.retrieve(query=query11)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query11, retrieved_chunks=retrieved_docs
)

answer

'Self-evolving claws can run more safely in clouds, on prem, RTX PCs, and DGX Spark.\nSource: NemoClaw Overview.md, Section: Overview of NVIDIA NemoClaw'

In [55]:
query12 = "What is the purpose of the openclaw-pricing policy preset introduced during onboarding in NemoClaw version v0.0.53"

In [56]:
retrieved_docs = retriever.retrieve(query=query12)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query12, retrieved_chunks=retrieved_docs
)

answer

'The `openclaw-pricing` policy preset, introduced in NemoClaw version v0.0.53, is used for model-pricing reference fetches from LiteLLM and OpenRouter. Its purpose is to allow session JSONL records to populate `usage.cost` without widening egress beyond these two read-only pricing endpoints.\nSource: NemoClaw Release Notes.md, Section: v0.0.53'

In [57]:
query13 = "Which commercially developed lunar landers are being tested for docking procedures in Earth orbit during the planned Artemis III mission"

In [58]:
retrieved_docs = retriever.retrieve(query=query13)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query13, retrieved_chunks=retrieved_docs
)

answer

"During the planned Artemis III mission, SpaceX's Starship HLS and Blue Origin's Blue Moon, both commercially developed lunar landers, are planned to be tested for rendezvous and docking procedures in Earth orbit.\n\nSource: Artemis Program.txt, None, None (Chunk 1)"

In [59]:
query14 = "Which specific Mars crater and mountain features is the Curiosity Rover exploring as part of the Mars Science Laboratory mission"

In [60]:
retrieved_docs = retriever.retrieve(query=query14)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query14, retrieved_chunks=retrieved_docs
)

answer

"The Curiosity Rover is exploring Gale crater and Mount Sharp on Mars as part of NASA's Mars Science Laboratory (MSL) mission. (Source: Curiosity Rover.txt, Chunk 1)"

In [61]:
query15 = "Who submitted the winning name proposal for the Curiosity Rover, and what special privilege did the winner receive during assembly"

In [62]:
retrieved_docs = retriever.retrieve(query=query15)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query15, retrieved_chunks=retrieved_docs
)

answer

"Twelve-year-old Clara Ma from Sunflower Elementary School in Lenexa, Kansas, submitted the winning name proposal for the Curiosity Rover. As her prize, she won a trip to NASA's Jet Propulsion Laboratory (JPL) in Pasadena, California, where she signed her name directly onto the rover as it was being assembled. (Source: Curiosity Rover.txt, Chunk 1)"

In [63]:
query16 = "Why does the James Webb Space Telescope create images of comparable resolution to Hubble despite having a mirror diameter that is 2.7 times larger"

In [64]:
retrieved_docs = retriever.retrieve(query=query16)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query16, retrieved_chunks=retrieved_docs
)

answer

'Despite having a mirror diameter 2.7 times larger than the Hubble Space Telescope, the James Webb Space Telescope produces images of comparable resolution because it observes in the infrared spectrum. The infrared spectrum has longer wavelengths than the visible spectrum observed by Hubble, and a larger information-gathering surface is required to achieve the desired resolution when observing longer wavelengths.\n(Source: James Web Telescope.txt, Chunk 1)'

In [65]:
query17 = "Around which specific outer space destination does the James Webb Space Telescope maintain its operational halo orbit, and how far is it from Earth"

In [66]:
retrieved_docs = retriever.retrieve(query=query17)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query17, retrieved_chunks=retrieved_docs
)

answer

'The James Webb Space Telescope maintains its operational halo orbit around the Sun–Earth L2 Lagrange point. It is approximately 1,500,000 km (930,000 mi) from Earth.\n\nSource: James Web Telescope.txt, Chunk 1, Chunk 2'

In [67]:
query18 = " What choice did NASA face regarding the planetary flyby trajectory for Voyager 1, and what factor determined the priority destination"

In [68]:
retrieved_docs = retriever.retrieve(query=query18)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query18, retrieved_chunks=retrieved_docs
)

answer

'NASA had to choose between a Pluto or Titan flyby for Voyager 1. The exploration of Titan took priority because it was known to have a substantial atmosphere. (Source: Voyager 1.txt, Chunk 1)'

In [69]:
query19 = "On what exact calendar date did Voyager 1 make history by crossing the heliopause to officially enter interstellar space"

In [72]:
retrieved_docs = retriever.retrieve(query=query19)

# 2. Generate final answer
answer = generator.generate_answer(
    question=query19, retrieved_chunks=retrieved_docs
)

answer

'Voyager 1 crossed the heliopause and entered interstellar space on August 25, 2012 (Source: Voyager 1.txt, Page: None, Section: None).'

# Testing session_pipeline

In [4]:
from pathlib import Path
from typing import List

import pymupdf4llm
from src.models.document import Document, DocumentMetadata

def load_pdf(file_path: Path) -> List[Document]:
    """
    Parses multi-column PDFs into clean markdown chunks per page 
    using layout heuristics. Zero GPU required.
    """
    pdf_path = Path(file_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    # page_chunks=True returns a list of dictionaries, one per page
    pages_data = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    documents: List[Document] = []

    for page in pages_data:
        text = page["text"]
        page_num = page["metadata"]["page_number"]
        
        if not text.strip():
            continue

        metadata = DocumentMetadata(
            source=pdf_path.name,
            file_type="pdf",
            page=page_num
        )
        
        documents.append(Document(page_content=text, metadata=metadata))
        
    return documents

In [ ]:
from src.core.chunker import Chunker
from src.core.embedding import GeminiEmbeddingGenerator
from src.utils.config import CHUNK_SIZE, CHUNK_OVERLAP, EMBEDDING_MODEL
from dotenv import load_dotenv
import os

chunker = Chunker(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embedder = GeminiEmbeddingGenerator(model_name=EMBEDDING_MODEL)

In [27]:
docs = load_pdf("RAG.pdf")
docs

[Document(page_content='RAG.md \n\n2026-05-08 \n\n1. Data Ingestion \n\n2. Chunking \n\n   - 🔹 Why do we need Chunking? \n\n   - 🔹 Example \n\n   - 🔹 What are Tokens? \n\n   - 🔹 Types of Chunking \n\n   1. Fixed-size Chunking \n\n   2. Recursive Chunking \n\n   3. Document-based / Structure-aware Chunking \n\n   4. Semantic Chunking \n\n   5. Sliding Window Chunking (Overlap) \n\n   - 🔹 Important Parameters \n\n   - 🔹 Tradeoffs \n\n   - 🔹 Best Practices \n\n3. Embeddings \n\n   - 🔹 Why do we need Embeddings in RAG? \n\n   - 🔹 What is a Vector? \n\n   - 🔹 Embedding Dimensions \n\n   - 🔹 Embedding Model Types — Two Levels \n\n   - 🔹 Level 1 — Base Architectures \n\n      1. BERT-style (Encoder-only Transformers) \n\n      2. Sentence Transformers (Fine-tuned BERT) \n\n      3. GPT-style (Decoder-only Transformers) \n\n      4. Proprietary / API-based Models Key API models: \n\n   - 🔹 Level 2 — Usage Patterns \n\n      1. Bi-encoder \n\n      2. Cross-encoder \n\n   - 🔹 How They Work Toge

In [28]:
chunked_docs = chunker.chunk_documents(docs)
chunked_docs

[Document(page_content='RAG.md  \n2026-05-08  \n1. Data Ingestion  \n2. Chunking  \n- 🔹 Why do we need Chunking?  \n- 🔹 Example  \n- 🔹 What are Tokens?  \n- 🔹 Types of Chunking  \n1. Fixed-size Chunking  \n2. Recursive Chunking  \n3. Document-based / Structure-aware Chunking  \n4. Semantic Chunking  \n5. Sliding Window Chunking (Overlap)  \n- 🔹 Important Parameters  \n- 🔹 Tradeoffs  \n- 🔹 Best Practices  \n3. Embeddings  \n- 🔹 Why do we need Embeddings in RAG?  \n- 🔹 What is a Vector?  \n- 🔹 Embedding Dimensions', metadata=DocumentMetadata(source='RAG.pdf', file_type='pdf', page=1, section=None, chunk_id='rag_page_1_chunk_0', chunk_index=0, parent_document_id='rag')),
 Document(page_content='- 🔹 Why do we need Embeddings in RAG?  \n- 🔹 What is a Vector?  \n- 🔹 Embedding Dimensions  \n- 🔹 Embedding Model Types — Two Levels  \n- 🔹 Level 1 — Base Architectures  \n1. BERT-style (Encoder-only Transformers)  \n2. Sentence Transformers (Fine-tuned BERT)  \n3. GPT-style (Decoder-only Transform

In [29]:
embedded_docs = embedder.embed_text(chunked_docs)

In [31]:
print("Notebook print test")

Notebook print test


In [ ]:
from groq import Groq
from src.utils.config import ROUTER_MODEL, ROUTER_TEMPERATURE

prompt = """
        You are a helpful RAG assistant. You have access to a knowledge base 
        of documents including tech specs, space mission records, and architecture docs.

        For greetings, small talk, or questions about your capabilities — respond naturally and briefly.
        When asked what you can do, explain that users can ask questions about the documents 
        in the knowledge base and you'll find and summarize relevant information.        
        """

client = Groq(api_key="")

In [ ]:
response = client.chat.completions.create(
    model=ROUTER_MODEL,
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": query}
        ],
    temperature=ROUTER_TEMPERATURE,
)

In [ ]:
from google import genai

client = genai.Client(api_key="")


In [5]:
text = "Hey, are you down to grab some pizza later? I'm starving!"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    config={
        "system_instruction": "Only output the translated text",
        "max_output_tokens": 100,
    },
    contents=f"Translate the following text to German: {text}", 
)

print(response.text)

Hey, hast du Lust, später Pizza essen zu gehen? Ich habe riesigen Hunger!


In [2]:
model = client.models.get(
    model="gemini-3.1-flash-lite"
)

print(model)

name='models/gemini-3.1-flash-lite' display_name='Gemini 3.1 Flash Lite' description='Gemini 3.1 Flash Lite' version='3.1-flash-lite-05-2026' endpoints=None labels=None tuned_model_info=TunedModelInfo() input_token_limit=1048576 output_token_limit=65536 supported_actions=['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent'] default_checkpoint_id=None checkpoints=None temperature=1.0 max_temperature=2.0 top_p=0.95 top_k=64 thinking=True


In [ ]:
import os 
from src.core.query_router import QueryRouter
from src.utils.config import ROUTER_MODEL, ROUTER_TEMPERATURE


router = QueryRouter(model=ROUTER_MODEL, temperature=ROUTER_TEMPERATURE)

In [7]:
query = "hello"
router.classify(query)

'CONVERSATIONAL'

In [3]:
query = "what is nemoclaw?"
router.classify(query)

'RETRIEVAL'

In [4]:
query = "What are you capable of?"
router.classify(query)

'CONVERSATIONAL'

In [5]:
query = "Hey, My name is dhanush, what is your name?"
router.classify(query)

'CONVERSATIONAL'

In [8]:
query = "Do you remember my name?"
router.classify(query)

'CONVERSATIONAL'

In [9]:
query = "what does the document say?"
router.classify(query)

'RETRIEVAL'